In [13]:
### import necessary packages
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.impute import KNNImputer
from sklearn.preprocessing import LabelEncoder
import numpy as np

In [14]:
### import data
data= pd.read_csv("./brca_metabric_clinical_data.tsv", sep="\t")

In [15]:
### feature selection

# list of featires to keep
keep_columns=["Age at Diagnosis", 
              "Cancer Type Detailed", 
              "Inferred Menopausal State", 
              "Tumor Size",
              "Lymph nodes examined positive",
              "3-Gene classifier subtype",
              "Pam50 + Claudin-low subtype",
              "Integrative Cluster",
              "Overall Survival (Months)"]

# remove other features
data = data[keep_columns].copy()

# confirm removal
print(data.columns)

# remove sapmeles where Overall Survival is na
data = data.dropna(subset=["Overall Survival (Months)"])

Index(['Age at Diagnosis', 'Cancer Type Detailed', 'Inferred Menopausal State',
       'Tumor Size', 'Lymph nodes examined positive',
       '3-Gene classifier subtype', 'Pam50 + Claudin-low subtype',
       'Integrative Cluster', 'Overall Survival (Months)'],
      dtype='str')


In [16]:
### group rare cancer types together

top3 = data["Cancer Type Detailed"].value_counts().nlargest(3).index

data["Cancer Type Grouped"] = data["Cancer Type Detailed"].apply(
    lambda x: x if x in top3 else "Other")

print(data["Cancer Type Grouped"].value_counts())

Cancer Type Grouped
Breast Invasive Ductal Carcinoma             1538
Breast Mixed Ductal and Lobular Carcinoma     211
Breast Invasive Lobular Carcinoma             146
Other                                          86
Name: count, dtype: int64


In [17]:
### discretize overall survival (months) into 4 groups

for index, row in data.iterrows():
    if row["Overall Survival (Months)"]>=180:
        data.loc[index, "Survival Duration"] = "15+ years"
    elif row["Overall Survival (Months)"]>=120:
        data.loc[index, "Survival Duration"] = "10-15 years"
    elif row["Overall Survival (Months)"]>=60:
        data.loc[index, "Survival Duration"] = "5-10 years"
    else:
        data.loc[index, "Survival Duration"] = "<5 years"

In [18]:
### split predictors and response

predictors=["Age at Diagnosis", 
              "Cancer Type Grouped", 
              "Inferred Menopausal State", 
              "Tumor Size",
              "Lymph nodes examined positive",
              "3-Gene classifier subtype",
              "Pam50 + Claudin-low subtype",
              "Integrative Cluster"]

x= data[predictors]
y=data["Survival Duration"]

In [19]:
### split into test and train data 
x_train, x_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42)

In [20]:
### impute missing predictor values with KNN imputation (k=5)

# check NaNs
print(x_train.isna().sum())
print(x_test.isna().sum())

# Fill single missing "Inferred Menopausal State" value with "Post"
x_train["Inferred Menopausal State"] = x_train["Inferred Menopausal State"].fillna("Post")
x_test["Inferred Menopausal State"] = x_test["Inferred Menopausal State"].fillna("Post")

# Encode categorical columns for KNN imputation
x_train_encoded = x_train.copy()
x_test_encoded = x_test.copy()

categorical_cols = ["Cancer Type Grouped", "3-Gene classifier subtype", 
                    "Pam50 + Claudin-low subtype", "Integrative Cluster"]
label_encoders = {}

for col in categorical_cols:
    le = LabelEncoder()
    # Fit on all unique values from both train and test
    all_values = pd.concat([x_train_encoded[col], x_test_encoded[col]]).dropna().unique()
    le.fit(all_values)
    label_encoders[col] = le    
    
    # Encode training and test data, preserving NaN values
    x_train_encoded[col] = x_train_encoded[col].apply(
        lambda x: le.transform([x])[0] if pd.notna(x) else np.nan)
    x_test_encoded[col] = x_test_encoded[col].apply(
        lambda x: le.transform([x])[0] if pd.notna(x) else np.nan)

# Apply KNN imputation with k=5 (only for columns with remaining missing values)
knn_imputer = KNNImputer(n_neighbors=5)
x_train_imputed = pd.DataFrame(
    knn_imputer.fit_transform(x_train_encoded), 
    columns=x_train_encoded.columns)
x_test_imputed = pd.DataFrame(
    knn_imputer.transform(x_test_encoded), 
    columns=x_test_encoded.columns)

# Decode categorical columns back to original values
for col in categorical_cols:
    le = label_encoders[col]
    x_train_imputed[col] = x_train_imputed[col].round().astype(int).apply(
        lambda x: le.inverse_transform([x])[0])
    x_test_imputed[col] = x_test_imputed[col].round().astype(int).apply(
        lambda x: le.inverse_transform([x])[0])

x_train = x_train_imputed
x_test = x_test_imputed

Age at Diagnosis                   0
Cancer Type Grouped                0
Inferred Menopausal State          0
Tumor Size                        19
Lymph nodes examined positive     63
3-Gene classifier subtype        172
Pam50 + Claudin-low subtype        0
Integrative Cluster                0
dtype: int64
Age at Diagnosis                  0
Cancer Type Grouped               0
Inferred Menopausal State         1
Tumor Size                        7
Lymph nodes examined positive    13
3-Gene classifier subtype        45
Pam50 + Claudin-low subtype       1
Integrative Cluster               1
dtype: int64


ValueError: could not convert string to float: 'Post'

In [ ]:
# confirm NaNs are removed
print(x_train.isna().sum())
print(x_test.isna().sum())

Age at Diagnosis                 0
Cancer Type Grouped              0
Inferred Menopausal State        0
Tumor Size                       0
Lymph nodes examined positive    0
3-Gene classifier subtype        0
Pam50 + Claudin-low subtype      0
Integrative Cluster              0
dtype: int64
Age at Diagnosis                 0
Cancer Type Grouped              0
Inferred Menopausal State        0
Tumor Size                       0
Lymph nodes examined positive    0
3-Gene classifier subtype        0
Pam50 + Claudin-low subtype      0
Integrative Cluster              0
dtype: int64


In [ ]:
### seperate x train and x test for each model

# model 1- baseline
x1=["Age at Diagnosis", 
    "Cancer Type Grouped", 
    "Inferred Menopausal State", 
    "Tumor Size",
    "Lymph nodes examined positive"]
x1_train= x_train[x1]
x1_test= x_test[x1]

# model 2- baseline + 3-gene
x2= x1 + ["3-Gene classifier subtype"]
x2_train= x_train[x2]
x2_test= x_test[x2]

# model 3- baseline + Pam 50
x3= x1 + ["Pam50 + Claudin-low subtype"]
x3_train= x_train[x3]
x3_test= x_test[x3]

# model 4 - baseline + integrative clusters
x4= x1 + ["Integrative Cluster"]
x4_train= x_train[x4]
x4_test= x_test[x4]



In [ ]:
### One hot encode

x1_train = pd.get_dummies(x1_train)
x1_test = pd.get_dummies(x1_test)

x2_train = pd.get_dummies(x2_train)
x2_test = pd.get_dummies(x2_test)

x3_train = pd.get_dummies(x3_train)
x3_test = pd.get_dummies(x3_test)

x4_train = pd.get_dummies(x4_train)
x4_test = pd.get_dummies(x4_test)

In [ ]:
### save datasets
x1_train.to_csv("./x1_train.csv", index=False)
x1_test.to_csv("./x1_test.csv", index=False)

x2_train.to_csv("./x2_train.csv", index=False)
x2_test.to_csv("./x2_test.csv", index=False)\

x3_train.to_csv("./x3_train.csv", index=False)
x3_test.to_csv("./x3_test.csv", index=False)

x4_train.to_csv("./x4_train.csv", index=False)
x4_test.to_csv("./x4_test.csv", index=False)

y_train.to_csv("./y_train.csv", index=False)
y_test.to_csv("./y_test.csv", index=False)